In [ ]:
# Notebook 03 - EXTRAÇÃO DAS FEATURES

import os
import numpy as np
import pandas as pd

# 1. Carregar o dataset intermediário limpo salvo no Notebook 02
caminho_interim = "../data/interim/df_interim_limpo.csv"
if not os.path.exists(caminho_interim):
  raise FileNotFoundError(
      "Execute o Notebook 02 primeiro para gerar o dataset limpo!"
  )

print("Carregando dados intermediários...")
df_interim = pd.read_csv(caminho_interim)
print(f"Total de registros carregados: {len(df_interim)}")


# 2. Função para calcular a BCEA (Bivariate Contour Ellipse Area)
def calcular_bcea(x, y, probabilidade=0.68):
  """Calcula a Área da Elipse de Contorno Bivariado para dados de Gaze.

  Parâmetros:
      x (array-like): Coordenadas X do olhar.
      y (array-like): Coordenadas Y do olhar.
      probabilidade (float): Intervalo de confiança (padrão 0.68 para 1 desvio
        padrão).
  """
  if len(x) < 2 or len(y) < 2:
    return 0.0

  # Desvios padrão das coordenadas X e Y
  sig_x = np.std(x, ddof=1)
  sig_y = np.std(y, ddof=1)

  # Coeficiente de correlação de Pearson entre X e Y
  correlacao = np.corrcoef(x, y)[0, 1]
  if np.isnan(correlacao):
    correlacao = 0.0

  # Constante baseada no nível de probabilidade (Chi-quadrado com 2 graus de liberdade)
  k = -2 * np.log(1 - probabilidade)

  # Fórmula da BCEA
  bcea = 2 * np.pi * sig_x * sig_y * np.sqrt(1 - correlacao**2) * k
  return bcea


# 3. Agrupamento por Sujeito, Janela Temporal e Target para Extração de Features
resultados_features = []
grupos = df_interim.groupby(["sujeito_id", "janela_id", "target"])

print("Extraindo features espaciais (BCEA e Médias)...")
for (sujeito, janela, target_val), grupo in grupos:
    x = grupo["GazeX"].values
    y = grupo["GazeY"].values

    bcea_val = calcular_bcea(x, y)
    mean_x = np.mean(x)
    mean_y = np.mean(y)

    resultados_features.append({
        "sujeito_id": sujeito,
        "janela_id": janela,
        "target": target_val,
        "BCEA": bcea_val,
        "Mean_X": mean_x,
        "Mean_Y": mean_y,
    })

# Converter lista de features em DataFrame final consolidado
df_features = pd.DataFrame(resultados_features)

print(
    f"\nFeatures extraídas com sucesso! Total de janelas processadas:"
    f" {len(df_features)}"
)
display(df_features.head())

# 4. Salvar o dataset pronto para a modelagem de Machine Learning (Parte 4)
os.makedirs("../data/processed", exist_ok=True)
caminho_saida_final = "../data/processed/df_mega.csv"
df_features.to_csv(caminho_saida_final, index=False)
print(f"Dataset final salvo com sucesso em '{caminho_saida_final}'.")